In [1]:
import pandas as pd
import pymongo # Python driver to interact with MongoDB, gives access to MongoDB client APIs
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv(override=True)

True

In [2]:
project_root = Path.cwd().parent
data_path = project_root / os.getenv("DATA_PATH")
df = pd.read_csv(data_path)
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [3]:
# df should be converted into dict before we push it to mongodb, since MongoDB stores data as documents (JSON-like dictionaries)

data = df.to_dict(orient='records')

In [4]:
DB_NAME = "Proj1"
COLLECTION_NAME = "Proj1-Data"
username = os.getenv("MONGODB_USER")
password = os.getenv("MONGODB_PASSWORD")
cluster = os.getenv("MONGODB_CLUSTER_CONFIG")
CONNECTION_URL = f"mongodb+srv://{username}:{password}@{cluster}"


"""
MongoDB organizes data like this:
MongoDB Server
   └── Database
         └── Collection
               └── Documents
CONNECTION_URL connects to a MongoDB Atlas cluster, found in get connection string in MongoDB
"""

'\nMongoDB organizes data like this:\nMongoDB Server\n   └── Database\n         └── Collection\n               └── Documents\nCONNECTION_URL connects to a MongoDB Atlas cluster, found in get connection string in MongoDB\n'

In [5]:
client = pymongo.MongoClient(CONNECTION_URL) # Creates a connection to the MongoDB server, MongoClient() connects to your cloud MongoDB cluster
data_base = client[DB_NAME] # Selects the database, If Proj1 does not exist, MongoDB creates it automatically when data is inserted
collection = data_base[COLLECTION_NAME] # Selects the collection inside the database, Collections store the actual documents

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [6]:
# Uploading data to MongoDB
rec = collection.insert_many(data)
"""
Uploads the list of dictionaries to MongoDB
Each dictionary becomes a MongoDB document
insert_many() inserts multiple records at once
"""

'\nUploads the list of dictionaries to MongoDB\nEach dictionary becomes a MongoDB document\ninsert_many() inserts multiple records at once\n'

In [8]:
# Load back data from mongodb

df = pd.DataFrame(list(collection.find())).drop(columns='_id', axis=0) # collection.find() returns a MongoDB cursor
df.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0


In [ ]:
## If you are getting timeout issue

# import certifi
# client = pymongo.MongoClient(CONNECTION_URL, tlsCAFile=certifi.where())
# # TO CREATE THE DATABASE
# data_base = client[DB_NAME]
# collection = data_base[COLLECTION_NAME]
# # TO INSERT DATA INTO THE COLLECTION
# rec = collection.insert_many(data)